# 单关节机器人：从 MuJoCo 到 PPO 与离线推理

这个笔记本在云端 Python 环境中建立**简化单关节仿真**，训练连续位置参考，比较简单基线，并检查 ONNX 和动作后处理。它不连接或控制任何真实电机。Colab 可能需要登录，安装依赖需要联网。


## 1. 写入示例工程
代码和 MJCF 模型随笔记本携带；无需另行下载仓库。


In [ ]:
from pathlib import Path
root = Path('joint_rl')
root.mkdir(exist_ok=True)
(root / 'requirements.txt').write_text('gymnasium==1.4.0\nmujoco==4.0.1\nstable-baselines3==2.9.2\nnumpy==2.3.3\ntorch==2.12.0\nonnx==1.21.0\nonnxruntime==1.26.0\nmatplotlib==3.10.9\n', encoding='utf-8')
(root / 'model.xml').write_text('<mujoco model="one_joint_teaching">\n  <!-- Teaching model only: values below are assumptions, not identified hardware. -->\n  <compiler angle="radian"/>\n  <option timestep="0.005" gravity="0 0 -9.81" integrator="implicitfast"/>\n  <default>\n    <joint limited="true" range="-1.05 1.05" damping="0.06" armature="0.002"/>\n    <geom contype="0" conaffinity="0"/>\n  </default>\n  <worldbody>\n    <geom name="ground" type="plane" size="1 1 .01" rgba=".2 .25 .3 1"/>\n    <body name="arm" pos="0 0 .55">\n      <joint name="joint" type="hinge" axis="0 1 0"/>\n      <geom name="link" type="capsule" fromto="0 0 0 0 0 -.32" size=".025" mass=".25" rgba=".2 .8 .6 1"/>\n    </body>\n  </worldbody>\n  <!-- Position target [rad] in ctrl, a simplified PD servo with torque saturation [N m]. -->\n  <actuator>\n    <position name="servo" joint="joint" kp="6" kv="0.25" ctrllimited="true" ctrlrange="-.9 .9" forcelimited="true" forcerange="-1.5 1.5"/>\n  </actuator>\n</mujoco>\n', encoding='utf-8')
(root / 'contract.py').write_text('"""Shared observation/action contract; no simulator, training framework or I/O."""\nimport numpy as np\n\nQ_COMMAND_LIMIT = 0.9       # rad, this teaching model\'s assumed command bound\nQ_FAIL_LIMIT = 1.04         # rad, this teaching model\'s termination threshold\nQD_SCALE = 5.0              # rad/s, speed input normalizer\nPOLICY_PERIOD_S = 0.02      # seconds\nMAX_REFERENCE_RATE = 3.0    # rad/s, a modeled setpoint slew limit\n\n\ndef observation(q: float, qd: float, target: float, previous_action: float) -> np.ndarray:\n    values = np.asarray([q, qd, target, previous_action], dtype=np.float64)\n    if not np.all(np.isfinite(values)):\n        raise ValueError("observation values must be finite")\n    return np.asarray([\n        np.clip(q / Q_COMMAND_LIMIT, -2.0, 2.0),\n        np.clip(qd / QD_SCALE, -1.0, 1.0),\n        np.clip(target / Q_COMMAND_LIMIT, -1.0, 1.0),\n        np.clip(previous_action, -1.0, 1.0),\n    ], dtype=np.float32)\n\n\ndef position_reference(action: float, previous_reference_rad: float) -> float:\n    """Bound the requested position and its per-cycle change."""\n    if not np.isfinite(action) or not np.isfinite(previous_reference_rad):\n        raise ValueError("action and previous reference must be finite")\n    requested = float(np.clip(action, -1.0, 1.0) * Q_COMMAND_LIMIT)\n    previous = float(np.clip(previous_reference_rad, -Q_COMMAND_LIMIT, Q_COMMAND_LIMIT))\n    max_delta = MAX_REFERENCE_RATE * POLICY_PERIOD_S\n    return float(np.clip(requested, previous-max_delta, previous+max_delta))\n', encoding='utf-8')
(root / 'joint_env.py').write_text('"""A small, explicit Gymnasium task using a real MuJoCo single-joint model."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport gymnasium as gym\nfrom gymnasium import spaces\nimport mujoco\nimport numpy as np\nfrom contract import Q_COMMAND_LIMIT, Q_FAIL_LIMIT, QD_SCALE, POLICY_PERIOD_S, observation, position_reference\n\nHERE = Path(__file__).resolve().parent\nN_SUBSTEPS = 4              # 4 * 0.005 s = 0.020 s per policy update, 50 Hz\nMAX_STEPS = 150             # time limit, i.e. truncation\n\n\nclass JointReachEnv(gym.Env):\n    """Reach and hold a target angle with an approximate position-servo plant."""\n\n    metadata = {"render_modes": []}\n\n    def __init__(self, xml_path: str | Path | None = None):\n        super().__init__()\n        self.model = mujoco.MjModel.from_xml_path(str(xml_path or HERE / "model.xml"))\n        if (self.model.nq, self.model.nv, self.model.nu) != (1, 1, 1):\n            raise ValueError("This task contract requires exactly one hinge and one actuator")\n        if self.model.jnt_type[0] != mujoco.mjtJoint.mjJNT_HINGE or not self.model.jnt_limited[0]:\n            raise ValueError("The one joint must be a limited hinge; observation units are radians")\n        if self.model.jnt_range[0,0] >= -Q_FAIL_LIMIT or self.model.jnt_range[0,1] <= Q_FAIL_LIMIT:\n            raise ValueError("Joint limits must enclose the simulation failure thresholds")\n        if (self.model.actuator_trntype[0] != mujoco.mjtTrn.mjTRN_JOINT\n                or self.model.actuator_trnid[0,0] != 0 or not np.isclose(self.model.actuator_gear[0,0],1)):\n            raise ValueError("The actuator must drive the single joint with unit gear")\n        if (not self.model.actuator_ctrllimited[0] or not self.model.actuator_forcelimited[0]\n                or self.model.actuator_ctrlrange[0,0] > -Q_COMMAND_LIMIT\n                or self.model.actuator_ctrlrange[0,1] < Q_COMMAND_LIMIT):\n            raise ValueError("The actuator must limit force and accept the reference range ±0.9 rad")\n        kp = self.model.actuator_gainprm[0,0]\n        if (self.model.actuator_gaintype[0] != mujoco.mjtGain.mjGAIN_FIXED\n                or self.model.actuator_biastype[0] != mujoco.mjtBias.mjBIAS_AFFINE\n                or kp <= 0 or not np.isclose(self.model.actuator_biasprm[0,1], -kp)\n                or self.model.actuator_biasprm[0,2] > 0):\n            raise ValueError("This example requires a position actuator with non-negative damping")\n        if not np.isclose(self.model.opt.timestep * N_SUBSTEPS, POLICY_PERIOD_S):\n            raise ValueError("MJCF must retain a 20 ms policy interval unless the contract is updated")\n        self.data = mujoco.MjData(self.model)\n        self.observation_space = spaces.Box(\n            low=np.asarray([-2, -1, -1, -1], dtype=np.float32),\n            high=np.asarray([2, 1, 1, 1], dtype=np.float32), dtype=np.float32)\n        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)\n        self.target = 0.0\n        self.previous_action = 0.0\n        self.count = 0\n\n    def _observation(self) -> np.ndarray:\n        return observation(float(self.data.qpos[0]), float(self.data.qvel[0]), self.target, self.previous_action)\n\n    def reset(self, *, seed: int | None = None, options=None):\n        super().reset(seed=seed)\n        mujoco.mj_resetData(self.model, self.data)\n        self.data.qpos[0] = self.np_random.uniform(-0.7, 0.7)\n        self.data.qvel[0] = self.np_random.uniform(-0.1, 0.1)\n        self.target = float(self.np_random.uniform(-0.75, 0.75))\n        self.data.ctrl[0] = float(self.data.qpos[0])\n        self.previous_action = float(self.data.ctrl[0] / Q_COMMAND_LIMIT)\n        self.count = 0\n        mujoco.mj_forward(self.model, self.data)\n        return self._observation(), self._info()\n\n    def _info(self) -> dict:\n        q = float(self.data.qpos[0])\n        return {"angle_rad": q, "velocity_rad_s": float(self.data.qvel[0]),\n                "target_rad": self.target, "error_rad": self.target-q,\n                "command_rad": float(self.data.ctrl[0]),\n                "actuator_torque_nm": float(self.data.actuator_force[0])}\n\n    def step(self, action):\n        a = np.asarray(action, dtype=np.float64).reshape(-1)\n        if a.shape != (1,) or not np.isfinite(a[0]):\n            raise ValueError("expected one finite normalized action")\n        a = float(np.clip(a[0], -1.0, 1.0))\n        self.data.ctrl[0] = position_reference(a, self.previous_action * Q_COMMAND_LIMIT)\n        for _ in range(N_SUBSTEPS):\n            mujoco.mj_step(self.model, self.data)\n        q = float(self.data.qpos[0]); qd = float(self.data.qvel[0])\n        if not np.isfinite(q) or not np.isfinite(qd):\n            raise RuntimeError("MuJoCo produced an invalid state")\n        error = self.target - q\n        reward = (-2.0 * error**2 - 0.01 * qd**2\n                  - 0.005 * (a-self.previous_action)**2\n                  + (0.05 if abs(error) < 0.05 else 0.0))\n        self.previous_action = float(self.data.ctrl[0] / Q_COMMAND_LIMIT)\n        self.count += 1\n        terminated = abs(q) > Q_FAIL_LIMIT or abs(qd) > 12.0\n        if terminated:\n            reward -= 10.0\n        truncated = self.count >= MAX_STEPS and not terminated\n        return self._observation(), float(reward), bool(terminated), bool(truncated), self._info()\n', encoding='utf-8')
(root / 'inspect_model.py').write_text('"""Inspect a single-joint MuJoCo teaching model using known simulated inputs.\n\nThis is a numerical simulation only. It never opens a port or controls hardware.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport mujoco\n\nfrom contract import MAX_REFERENCE_RATE, POLICY_PERIOD_S, Q_COMMAND_LIMIT, position_reference\nfrom joint_env import HERE, N_SUBSTEPS, JointReachEnv\n\n\ndef simulate(env: JointReachEnv, requested_reference: float, cycles: int) -> dict:\n    """Apply one signed teaching reference to a fresh, zero-angle simulation."""\n    mujoco.mj_resetData(env.model, env.data)\n    env.data.qpos[0] = 0.0\n    env.data.qvel[0] = 0.0\n    env.data.ctrl[0] = 0.0\n    mujoco.mj_forward(env.model, env.data)\n    action = requested_reference / Q_COMMAND_LIMIT\n    reference = 0.0\n    rows = []\n    peak_force = 0.0\n    maximum_reference_change = 0.0\n    for i in range(cycles):\n        old_reference = reference\n        reference = position_reference(action, reference)\n        maximum_reference_change = max(maximum_reference_change, abs(reference-old_reference))\n        env.data.ctrl[0] = reference\n        for _ in range(N_SUBSTEPS):\n            mujoco.mj_step(env.model, env.data)\n            peak_force = max(peak_force, abs(float(env.data.actuator_force[0])))\n        rows.append({"cycle": i + 1, "simulated_time_s": round((i + 1) * POLICY_PERIOD_S, 6),\n                     "applied_reference_rad": float(reference), "angle_rad": float(env.data.qpos[0]),\n                     "velocity_rad_s": float(env.data.qvel[0]), "actuator_torque_nm": float(env.data.actuator_force[0])})\n    return {"requested_reference_rad": requested_reference, "max_reference_change_rad": maximum_reference_change,\n            "peak_abs_actuator_torque_nm_sampled_each_physics_step": peak_force, "first_sample": rows[0],\n            "last_sample": rows[-1], "samples": rows}\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--xml", type=Path, default=HERE / "model.xml")\n    parser.add_argument("--target", type=float, default=0.4, help="positive simulated reference in rad")\n    parser.add_argument("--cycles", type=int, default=20, help="number of 20 ms simulation cycles")\n    parser.add_argument("--output", type=Path, help="optional JSON result path")\n    args = parser.parse_args()\n    if not 0 < args.target <= Q_COMMAND_LIMIT or not 1 <= args.cycles <= 150:\n        parser.error("--target must be in (0, 0.9] rad; --cycles must be in [1, 150]")\n    env = JointReachEnv(args.xml)\n    try:\n        positive = simulate(env, args.target, args.cycles)\n        negative = simulate(env, -args.target, args.cycles)\n        result = {\n            "scope": "MuJoCo simulation only; no hardware connected or commanded",\n            "model": args.xml.name,\n            "physics_timestep_s": float(env.model.opt.timestep),\n            "physics_steps_per_control_cycle": N_SUBSTEPS,\n            "policy_period_s": POLICY_PERIOD_S,\n            "actuator_control_range_rad": env.model.actuator_ctrlrange[0].tolist(),\n            "actuator_force_range_nm": env.model.actuator_forcerange[0].tolist(),\n            "reference_change_limit_rad_per_cycle": MAX_REFERENCE_RATE * POLICY_PERIOD_S,\n            "positive": positive,\n            "negative": negative,\n            "direction_check": {\n                "positive_final_angle_is_positive": positive["last_sample"]["angle_rad"] > 0,\n                "negative_final_angle_is_negative": negative["last_sample"]["angle_rad"] < 0,\n            },\n            "limitations": "This checks two simulated inputs only, not real geometry, calibration, safety or hardware response.",\n        }\n        text = json.dumps(result, ensure_ascii=False, indent=2)\n        if args.output:\n            args.output.parent.mkdir(parents=True, exist_ok=True)\n            args.output.write_text(text + "\\n", encoding="utf-8")\n        summary = {\n            "scope": result["scope"], "model": result["model"],\n            "physics_timestep_s": result["physics_timestep_s"],\n            "physics_steps_per_control_cycle": result["physics_steps_per_control_cycle"],\n            "policy_period_s": result["policy_period_s"],\n            "actuator_control_range_rad": result["actuator_control_range_rad"],\n            "actuator_force_range_nm": result["actuator_force_range_nm"],\n            "reference_change_limit_rad_per_cycle": result["reference_change_limit_rad_per_cycle"],\n            "positive": {k:positive[k] for k in ("requested_reference_rad", "max_reference_change_rad", "peak_abs_actuator_torque_nm_sampled_each_physics_step", "first_sample", "last_sample")},\n            "negative": {k:negative[k] for k in ("requested_reference_rad", "max_reference_change_rad", "peak_abs_actuator_torque_nm_sampled_each_physics_step", "first_sample", "last_sample")},\n            "direction_check": result["direction_check"], "limitations": result["limitations"],\n            "detailed_log": str(args.output) if args.output else "not saved; provide --output to save all samples",\n        }\n        print(json.dumps(summary, ensure_ascii=False, indent=2))\n    finally:\n        env.close()\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(root / 'train.py').write_text('"""Train the continuous-action joint policy in simulation only."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport torch\nfrom stable_baselines3 import PPO\nfrom stable_baselines3.common.env_checker import check_env\nfrom joint_env import HERE, JointReachEnv\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--timesteps", type=int, default=100000)\n    parser.add_argument("--seed", type=int, default=123)\n    parser.add_argument("--output", type=Path, default=Path("artifacts"))\n    parser.add_argument("--xml", type=Path, default=HERE / "model.xml")\n    parser.add_argument("--quiet", action="store_true")\n    args = parser.parse_args()\n    if args.timesteps < 1:\n        parser.error("--timesteps must be positive")\n    args.output.mkdir(parents=True, exist_ok=True)\n    torch.set_num_threads(1)\n    env = JointReachEnv(args.xml)\n    check_env(env, warn=True)\n    model = PPO("MlpPolicy", env, seed=args.seed, device="cpu", verbose=0 if args.quiet else 1,\n                learning_rate=3e-4, n_steps=512, batch_size=64, n_epochs=8,\n                gamma=0.99, gae_lambda=0.95,\n                policy_kwargs=dict(net_arch=dict(pi=[64, 64], vf=[64, 64])))\n    model.learn(total_timesteps=args.timesteps, progress_bar=False)\n    model.save(args.output / "joint_ppo.zip")\n    (args.output / "training_config.json").write_text(json.dumps({\n        "algorithm":"Stable-Baselines3 PPO", "seed":args.seed,\n        "requested_timesteps":args.timesteps, "actual_timesteps":model.num_timesteps,\n        "model_xml_sha256":hashlib.sha256(args.xml.read_bytes()).hexdigest(),\n        "control_period_s":0.02, "obs":["q_rad/0.9", "clip(qd_rad_s/5)", "target_rad/0.9", "previous_applied_position_reference_rad/0.9"],\n        "action":"clip(policy_output,-1,1) * 0.9 rad, with a +/-0.06 rad/cycle setpoint slew limit",\n        "physical_validation":"none: simulation only"\n    }, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(f"Saved simulated policy: {args.output / \'joint_ppo.zip\'}")\n    env.close()\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(root / 'evaluate.py').write_text('"""Compare policies on the same 20 fixed, training-independent simulation seeds."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom stable_baselines3 import PPO\nfrom joint_env import HERE, JointReachEnv, MAX_STEPS\n\n\ndef evaluate(policy_name, model, episodes, first_seed, xml):\n    env = JointReachEnv(xml)\n    rows = []\n    records = []\n    for i in range(episodes):\n        seed = first_seed + i\n        obs, _ = env.reset(seed=seed)\n        random = np.random.default_rng(seed + 1000000)\n        errors = []\n        returns = 0.0\n        terminated = False\n        peak_torque = 0.0\n        for t in range(MAX_STEPS):\n            if policy_name == "zero":\n                action = np.asarray([0.0], dtype=np.float32)\n            elif policy_name == "random":\n                action = np.asarray([random.uniform(-1, 1)], dtype=np.float32)\n            elif policy_name == "target_copy":\n                action = np.asarray([obs[2]], dtype=np.float32)\n            else:\n                action, _ = model.predict(obs, deterministic=True)\n            obs, reward, terminated, truncated, info = env.step(action)\n            errors.append(abs(info["error_rad"]))\n            returns += reward\n            peak_torque = max(peak_torque, abs(info["actuator_torque_nm"]))\n            if i == 0:\n                records.append({"policy":policy_name, "seed":seed, "step":t+1,\n                    "time_s":round((t+1)*.02, 4), "target_rad":info["target_rad"],\n                    "angle_rad":info["angle_rad"], "velocity_rad_s":info["velocity_rad_s"],\n                    "command_rad":info["command_rad"], "actuator_torque_nm":info["actuator_torque_nm"]})\n            if terminated or truncated:\n                break\n        success = not terminated and len(errors) == MAX_STEPS and all(v < .05 for v in errors[-20:])\n        rows.append({"seed":seed, "mean_error_rad":float(np.mean(errors)),\n            "final_50_mean_error_rad":float(np.mean(errors[-50:])), "success":success,\n            "terminated":terminated, "episode_length":len(errors),\n            "episode_return":float(returns), "max_torque_nm":peak_torque})\n    env.close()\n    aggregate = {\n        "episodes":episodes, "mean_abs_error_rad":float(np.mean([r["mean_error_rad"] for r in rows])),\n        "final_50_mean_abs_error_rad":float(np.mean([r["final_50_mean_error_rad"] for r in rows])),\n        "final_50_mean_abs_error_deg":float(np.degrees(np.mean([r["final_50_mean_error_rad"] for r in rows]))),\n        "success_count":int(sum(r["success"] for r in rows)),\n        "success_fraction":float(np.mean([r["success"] for r in rows])),\n        "terminated_count":int(sum(r["terminated"] for r in rows)),\n        "mean_episode_return":float(np.mean([r["episode_return"] for r in rows])),\n        "max_observed_torque_nm":float(max(r["max_torque_nm"] for r in rows)),\n        "per_episode":rows,\n    }\n    return aggregate, records\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", type=Path, default=Path("artifacts/joint_ppo.zip"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts"))\n    parser.add_argument("--episodes", type=int, default=20)\n    parser.add_argument("--first-seed", type=int, default=20000)\n    parser.add_argument("--xml", type=Path, default=HERE / "model.xml")\n    parser.add_argument("--allow-model-mismatch", action="store_true",\n                        help="Allow an intentionally different XML for sim-to-sim comparison")\n    args = parser.parse_args()\n    if args.episodes < 1:\n        parser.error("--episodes must be positive")\n    args.output.mkdir(parents=True, exist_ok=True)\n    model_hash = hashlib.sha256(args.xml.read_bytes()).hexdigest()\n    config_path = args.model.parent/"training_config.json"\n    trained_hash = json.loads(config_path.read_text(encoding="utf-8")).get("model_xml_sha256") if config_path.exists() else None\n    same_model = model_hash == trained_hash if trained_hash else False\n    if not same_model and not args.allow_model_mismatch:\n        parser.error("MJCF hash does not match the model training config (or config is missing); retrain, or use --allow-model-mismatch for a deliberately labeled sim-to-sim test")\n    torch.set_num_threads(1)\n    trained = PPO.load(args.model, device="cpu")\n    # Same architecture and seed as train.py, with no learning.\n    untrained = PPO("MlpPolicy", JointReachEnv(args.xml), seed=123, device="cpu", n_steps=512,\n                    batch_size=64, n_epochs=8,\n                    policy_kwargs=dict(net_arch=dict(pi=[64, 64], vf=[64, 64])))\n    results = {"description":"Simulation only; identical independent initial conditions for each policy.",\n               "model_xml_sha256":model_hash, "same_xml_as_training":same_model,\n               "seed_range":[args.first_seed, args.first_seed+args.episodes-1],\n               "success_definition":"all last 20 steps |target-angle| < 0.05 rad and no early termination",\n               "baselines":{"zero":"always commands 0 rad", "random":"uniform random action each step",\n                            "target_copy":"hand-coded proportional reference equals target; no learning",\n                            "untrained":"fresh neural network before PPO updates", "ppo":"trained policy"},\n               "policies":{}}\n    all_traces = []\n    for name, model in [("zero",None),("random",None),("target_copy",None),("untrained",untrained),("ppo",trained)]:\n        summary, trace = evaluate(name, model, args.episodes, args.first_seed, args.xml)\n        results["policies"][name] = summary\n        all_traces.extend(trace)\n        print(f"{name:12} final50 MAE={summary[\'final_50_mean_abs_error_rad\']:.4f} rad "\n              f"({summary[\'final_50_mean_abs_error_deg\']:.2f} deg), success={summary[\'success_count\']}/{args.episodes}, "\n              f"terminated={summary[\'terminated_count\']}")\n    (args.output/"evaluation.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")\n    with (args.output/"trace_first_episode.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=list(all_traces[0].keys()))\n        writer.writeheader(); writer.writerows(all_traces)\n    print(f"Saved metrics and first-seed traces to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(root / 'export_onnx.py').write_text('"""Export the deterministic actor and verify against SB3\'s actual predict()."""\nfrom __future__ import annotations\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport onnx\nimport onnxruntime as ort\nimport torch\nfrom stable_baselines3 import PPO\n\n\nclass DeterministicActor(torch.nn.Module):\n    def __init__(self, policy):\n        super().__init__()\n        self.policy = policy\n\n    def forward(self, obs):\n        features = self.policy.extract_features(obs)\n        latent = self.policy.mlp_extractor.forward_actor(features)\n        # SB3 predict() clips this unsquashed Gaussian mean to the Box action space.\n        return torch.clamp(self.policy.action_net(latent), -1.0, 1.0)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", type=Path, default=Path("artifacts/joint_ppo.zip"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts"))\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    torch.set_num_threads(1)\n    model = PPO.load(args.model, device="cpu")\n    model.policy.set_training_mode(False)\n    actor = DeterministicActor(model.policy).eval()\n    path = args.output/"joint_actor.onnx"\n    torch.onnx.export(actor, torch.zeros((1,4), dtype=torch.float32), str(path),\n                      input_names=["observation"], output_names=["action"],\n                      dynamic_axes={"observation":{0:"batch"}, "action":{0:"batch"}},\n                      opset_version=17, dynamo=False)\n    onnx.checker.check_model(onnx.load(path))\n    options = ort.SessionOptions(); options.intra_op_num_threads = 1; options.inter_op_num_threads = 1\n    session = ort.InferenceSession(str(path), sess_options=options, providers=["CPUExecutionProvider"])\n    rng = np.random.default_rng(73913)\n    observations = rng.uniform([-1.2,-1,-.84,-1], [1.2,1,.84,1], size=(100,4)).astype(np.float32)\n    sb3_actions, _ = model.predict(observations, deterministic=True)\n    onnx_actions = session.run(["action"], {"observation":observations})[0]\n    difference = float(np.max(np.abs(sb3_actions - onnx_actions)))\n    result = {"inputs":"float32 observation [batch,4]", "outputs":"float32 clipped action [batch,1]",\n              "samples":len(observations), "max_abs_action_difference":difference,\n              "tolerance":1e-5, "passed":bool(difference <= 1e-5),\n              "warning":"This validates numerical inference only, not a hardware deployment."}\n    (args.output/"onnx_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps(result, ensure_ascii=False, indent=2))\n    if not result["passed"]:\n        raise SystemExit("ONNX and SB3 outputs differ beyond tolerance")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(root / 'deploy_offline.py').write_text('"""Pure offline inference/adapter example. No serial, bus, hardware or motor commands."""\nfrom __future__ import annotations\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport onnxruntime as ort\nfrom contract import Q_COMMAND_LIMIT, Q_FAIL_LIMIT, POLICY_PERIOD_S, observation, position_reference\n\n\nclass OfflineJointPolicy:\n    """Demonstrates policy I/O only. Thresholds are teaching-model assumptions."""\n    def __init__(self, model_path: str | Path):\n        options = ort.SessionOptions(); options.intra_op_num_threads = 1; options.inter_op_num_threads = 1\n        self.session = ort.InferenceSession(str(model_path), sess_options=options,\n                                            providers=["CPUExecutionProvider"])\n        self.previous_action = None\n\n    def reset(self, current_angle_rad: float):\n        # Initial reference equals the observed angle, as in training; sends no command.\n        if not np.isfinite(current_angle_rad) or abs(current_angle_rad) > Q_COMMAND_LIMIT:\n            raise ValueError("initial angle outside the teaching model\'s command range")\n        self.previous_action = current_angle_rad / Q_COMMAND_LIMIT\n\n    def infer(self, *, angle_rad: float, velocity_rad_s: float,\n              target_rad: float, sample_age_s: float) -> dict:\n        values = np.asarray([angle_rad, velocity_rad_s, target_rad, sample_age_s], dtype=float)\n        if self.previous_action is None:\n            raise ValueError("reset with an initial measured angle before inference")\n        if not np.all(np.isfinite(values)):\n            raise ValueError("invalid observation or timestamp")\n        if not 0.0 <= sample_age_s <= 2 * POLICY_PERIOD_S:\n            raise ValueError("stale sample: no new reference produced")\n        if abs(angle_rad) > Q_FAIL_LIMIT or abs(velocity_rad_s) > 12 or abs(target_rad) > .75:\n            raise ValueError("outside the teaching model\'s tested operating envelope")\n        features = observation(angle_rad, velocity_rad_s, target_rad, self.previous_action)\n        prediction = self.session.run(["action"], {"observation":features.reshape(1,4)})[0]\n        action = float(prediction[0,0])\n        if not np.isfinite(action):\n            raise ValueError("invalid policy output")\n        action = float(np.clip(action, -1.0, 1.0))\n        reference = position_reference(action, self.previous_action * Q_COMMAND_LIMIT)\n        self.previous_action = reference / Q_COMMAND_LIMIT\n        return {"observation":features.tolist(), "action_normalized":action,\n                "position_reference_rad":reference, "control_period_s":POLICY_PERIOD_S,\n                "executed_on_hardware":False}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", type=Path, default=Path("artifacts/joint_actor.onnx"))\n    args = parser.parse_args()\n    policy = OfflineJointPolicy(args.model)\n    # These inputs are invented for demonstration and do not come from a device.\n    policy.reset(current_angle_rad=0.10)\n    result = policy.infer(angle_rad=0.10, velocity_rad_s=0.0, target_rad=0.50, sample_age_s=0.0)\n    print(json.dumps(result, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(root / 'check_deployment.py').write_text('"""Verify SB3 / ONNX / position reference parity in the simulator; no hardware."""\nfrom __future__ import annotations\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom stable_baselines3 import PPO\nfrom deploy_offline import OfflineJointPolicy\nfrom joint_env import HERE, JointReachEnv\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--artifacts", type=Path, default=Path("artifacts"))\n    parser.add_argument("--xml", type=Path, default=HERE / "model.xml")\n    args = parser.parse_args()\n    torch.set_num_threads(1)\n    model = PPO.load(args.artifacts/"joint_ppo.zip", device="cpu")\n    adapter = OfflineJointPolicy(args.artifacts/"joint_actor.onnx")\n    env = JointReachEnv(args.xml)\n    max_action_diff = max_ref_diff = max_obs_diff = 0.0\n    samples = 0\n    for seed in range(40000, 40010):\n        obs, info = env.reset(seed=seed)\n        adapter.reset(info["angle_rad"])\n        for _ in range(150):\n            action, _ = model.predict(obs, deterministic=True)\n            result = adapter.infer(angle_rad=info["angle_rad"], velocity_rad_s=info["velocity_rad_s"],\n                                   target_rad=info["target_rad"], sample_age_s=0.0)\n            max_obs_diff = max(max_obs_diff, float(np.max(np.abs(obs-np.asarray(result["observation"])))))\n            max_action_diff = max(max_action_diff, abs(float(action[0])-result["action_normalized"]))\n            obs, _, terminated, truncated, info = env.step(action)\n            max_ref_diff = max(max_ref_diff, abs(info["command_rad"]-result["position_reference_rad"]))\n            samples += 1\n            if terminated or truncated:\n                break\n    reject_count = 0\n    for overrides in [{"sample_age_s":.041}, {"target_rad":.8}, {"angle_rad":float("nan")}]:\n        sample = {"angle_rad":0.1, "velocity_rad_s":0.0, "target_rad":0.5, "sample_age_s":0.0}\n        sample.update(overrides)\n        try:\n            adapter.infer(**sample)\n        except ValueError:\n            reject_count += 1\n    result = {"simulation_episodes":10, "control_samples":samples,\n              "max_observation_difference":max_obs_diff, "max_action_difference":max_action_diff,\n              "max_reference_difference_rad":max_ref_diff,\n              "tolerances":{"observation":1e-5, "action":1e-4, "reference_rad":1e-5},\n              "rejected_invalid_samples":reject_count, "expected_rejections":3,\n              "passed":bool(max_obs_diff < 1e-5 and max_action_diff < 1e-4 and max_ref_diff < 1e-5 and reject_count == 3),\n              "hardware_connected":False}\n    (args.artifacts/"deployment_contract_check.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps(result, ensure_ascii=False, indent=2))\n    if not result["passed"]:\n        raise SystemExit("Contract check failed")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(root / 'plot_results.py').write_text('"""Plot measured simulated evaluation output. No fabricated example values."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib import font_manager\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--artifacts", type=Path, default=Path("artifacts"))\n    args = parser.parse_args()\n    results = json.loads((args.artifacts/"evaluation.json").read_text(encoding="utf-8"))\n    with (args.artifacts/"trace_first_episode.csv").open(encoding="utf-8", newline="") as f:\n        rows = list(csv.DictReader(f))\n    available = {item.name for item in font_manager.fontManager.ttflist}\n    cjk = next((name for name in ("Microsoft YaHei", "Noto Sans CJK SC", "Noto Sans CJK JP", "SimHei") if name in available), None)\n    plt.rcParams["font.family"] = [cjk, "DejaVu Sans"] if cjk else ["DejaVu Sans"]\n    plt.rcParams["axes.unicode_minus"] = False\n    text = lambda zh, en: zh if cjk else en\n    fig, axes = plt.subplots(1,2,figsize=(13,5), constrained_layout=True)\n    fig.suptitle(text("单关节位置跟踪：MuJoCo 仿真结果", "Single-joint position tracking: MuJoCo simulation"), fontsize=18, weight="bold")\n    labels=[text("零动作","Zero"), text("未训练网络","Untrained"), "PPO", text("直接给目标","Direct target")]\n    keys=["zero", "untrained", "ppo", "target_copy"]\n    values=[results["policies"][k]["final_50_mean_abs_error_deg"] for k in keys]\n    axes[0].bar(labels, values, color=["#8ba1b0", "#8ba1b0", "#20a77e", "#e6a149"])\n    axes[0].set_ylabel(text("最后 50 步平均绝对误差（度）", "Mean absolute error, final 50 steps (deg)"))\n    axes[0].set_title(text("20 个独立初始条件 · 越低越好", "20 independent initial conditions · lower is better"))\n    for i,v in enumerate(values): axes[0].text(i,v+0.3,f"{v:.2f}°",ha="center",fontsize=10)\n    for name, label, color in [("zero",text("零动作","Zero"),"#8ba1b0"),("ppo","PPO","#20a77e"),("target_copy",text("直接给目标","Direct target"),"#e6a149")]:\n        selected=[r for r in rows if r["policy"]==name]\n        axes[1].plot([float(r["time_s"]) for r in selected],[float(r["angle_rad"]) for r in selected],label=label,color=color)\n    target=float(rows[0]["target_rad"])\n    axes[1].axhline(target,linestyle="--",color="#4c5360",label=text("目标角","Target"))\n    axes[1].set_xlabel(text("时间（秒）","Time (s)"));axes[1].set_ylabel(text("关节角（弧度）","Joint angle (rad)"))\n    axes[1].set_title(text(f"第一组初始条件 · 种子 {rows[0][\'seed\']}", f"First initial condition · seed {rows[0][\'seed\']}"))\n    axes[1].legend(frameon=False)\n    fig.text(.5,.01,text("仅为仿真；简单任务中直接给目标是应当保留的强基线。", "Simulation only; a direct target is a useful baseline for a simple task."),ha="center",color="#63727c",fontsize=10)\n    fig.savefig(args.artifacts/"tracking_comparison.png", dpi=160, bbox_inches="tight")\n    print(f"Saved {args.artifacts/\'tracking_comparison.png\'}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
%pip install -q -r joint_rl/requirements.txt


## 模型自检：先验证方向和限幅
这个检查只运行 MuJoCo，分别对模拟关节施加正负位置参考，记录参考变化和每个物理步的力矩；不连接硬件。


In [ ]:
!python joint_rl/inspect_model.py --output joint_rl/model_inspection.json


## 2. 训练连续动作策略
模型每 20 ms 输出一次归一化动作；实际位置参考还经过限幅与变化率限制。这里是单关节简化模型，不是任何真实舵机的参数。


In [ ]:
!python joint_rl/train.py --timesteps 100000 --seed 123 --output joint_rl/my_run


## 3. 独立评估并对照简单基线
20 个未用于训练的初始种子；成功指完整回合不提前终止，且最后 20 步每一步误差小于 0.05 rad。不要把这个数字当作真机成功率。


In [ ]:
!python joint_rl/evaluate.py --model joint_rl/my_run/joint_ppo.zip --output joint_rl/my_run
!python joint_rl/plot_results.py --artifacts joint_rl/my_run


In [ ]:
from IPython.display import Image, display
display(Image(filename='joint_rl/my_run/tracking_comparison.png'))


## 4. 导出并核对输入、输出与限速
ONNX 图中包含动作夹紧；角度映射和每周期位置参考变化率限制在图外。相同输入的数值对照通过，不等于真实硬件已验证。


In [ ]:
!python joint_rl/export_onnx.py --model joint_rl/my_run/joint_ppo.zip --output joint_rl/my_run
!python joint_rl/check_deployment.py --artifacts joint_rl/my_run
!python joint_rl/deploy_offline.py --model joint_rl/my_run/joint_actor.onnx


## 5. 打包结果
在 Colab 可下载；其他 Jupyter 环境直接查看 `joint_rl/my_run/`。更换真实机器人模型、接口或主控时，需要重新核对并验证整套输入输出契约。


In [ ]:
import shutil
shutil.make_archive('我的单关节仿真结果', 'zip', 'joint_rl/my_run')
try:
    from google.colab import files
    files.download('我的单关节仿真结果.zip')
except ImportError:
    print('已生成 我的单关节仿真结果.zip')
